In [35]:
import numpy as np
from iodata import load_one
from gbasis.wrappers import from_iodata
from gbasis.integrals.overlap import overlap_integral
from gbasis.evals.density import *
from grid import GaussChebyshev, BeckeRTransform, MolGrid, BeckeWeights

In [36]:
mol = load_one("../examples/water.fchk")

In [37]:
# 1-electron reduced density matrix (1-RDM) from SCF calculation
rdm = mol.one_rdms.get("scf")


In [38]:
coeffs = mol.mo.coeffs
occs = mol.mo.occs
rdm_raw = np.dot(coeffs * occs, coeffs.T)

In [39]:
mo = mol.mo
# Spin-resolved density matrices
# Alpha
rdm_a = (mo.coeffsa * mo.occsa) @ mo.coeffsa.conj().T

# Beta
rdm_b = (mo.coeffsb * mo.occsb) @ mo.coeffsb.conj().T

# Total density
rdm_total = rdm_a + rdm_b

# Verify consistency
rdm_raw = (mo.coeffs * mo.occs) @ mo.coeffs.conj().T

assert np.allclose(rdm_total, rdm_raw)
assert np.allclose(rdm_total, rdm)

In [40]:
np.allclose(rdm_raw, rdm)

True

In [41]:
# basis function from mol object
basis = from_iodata(mol)

In [42]:
# basis function from mol object
basis = from_iodata(mol)

# AO overlap matrix
S = overlap_integral(basis)

# Number of electrons considering non-orthogonal basis
n_alpha = np.trace(rdm_a @ S)
n_beta = np.trace(rdm_b @ S)

print("Alpha electrons:", n_alpha)
print("Beta electrons:", n_beta)

# Check AO orthonormality
print(np.allclose(mo.coeffsa.T @ S @ mo.coeffsa,
                  np.eye(mo.coeffsa.shape[1])))

Alpha electrons: 4.999999998010127
Beta electrons: 4.999999998010127
False


In [43]:
C = mo.coeffsa
occupied = mo.occsa > 1e-8

C_occ = C[:, occupied]
C_vir = C[:, ~occupied]

S_oo = C_occ.T @ S @ C_occ
S_vv = C_vir.T @ S @ C_vir
S_ov = C_occ.T @ S @ C_vir

print("Occupied-occupied:",
      np.max(np.abs(S_oo - np.eye(S_oo.shape[0]))))

print("Virtual-virtual:",
      np.max(np.abs(S_vv - np.eye(S_vv.shape[0]))))

print("Occupied-virtual:",
      np.max(np.abs(S_ov)))


Occupied-occupied: 9.384191201888825e-10
Virtual-virtual: 2.001773047163264e-08
Occupied-virtual: 1.9997340672262e-08


In [44]:
# Construct molecular grid
oned = GaussChebyshev(npoints=30)
rgrid = BeckeRTransform(0.0, R=1.5).transform_1d_grid(oned)

grid = MolGrid.from_preset(
    atnums=mol.atnums,
    atcoords=mol.atcoords,
    rgrid=rgrid,
    preset="coarse",
    aim_weights=BeckeWeights(),
    store=True,
)

# Evaluate spin-resolved densities
dens_a = evaluate_density(rdm_a, basis, grid.points)
dens_b = evaluate_density(rdm_b, basis, grid.points)

# Total electron density
moldens_raw = dens_a + dens_b
moldens = evaluate_density(rdm, basis, grid.points)

# Spin density
spin_dens = dens_a - dens_b

In [45]:
print("Alpha electrons:", grid.integrate(dens_a))
print("Beta electrons:", grid.integrate(dens_b))
print("Total electrons:", grid.integrate(moldens))
print("Total electrons (raw):", grid.integrate(moldens_raw))
print("Spin polarization:", grid.integrate(spin_dens))

Alpha electrons: 5.000012844028549
Beta electrons: 5.000012844028549
Total electrons: 10.000025689713967
Total electrons (raw): 10.000025688057098
Spin polarization: 0.0


In [46]:
spin_dens

array([0., 0., 0., ..., 0., 0., 0.], shape=(3460,))

In [47]:
print("MO type:", mo.kind)
print("Maximum spin density:", np.max(spin_dens))
print("Minimum spin density:", np.min(spin_dens))
print("Integrated absolute spin density:",
      grid.integrate(np.abs(spin_dens)))

MO type: unrestricted
Maximum spin density: 0.0
Minimum spin density: 0.0
Integrated absolute spin density: 0.0


In [48]:
spin_dens_true = mol.one_rdms.get("scf_spin")
spin_dens_true = evaluate_density(spin_dens_true, basis, grid.points)
print("Integrated absolute spin density:",
      grid.integrate(np.abs(spin_dens_true)))

Integrated absolute spin density: 0.0


### Test Difference between Restricted and Unrestricted Functionals

In [49]:
res_mol = load_one("../examples/0001_q000_m01_k00_force_b3lyp_def2tzvpd.fchk")
unr_mol = load_one("../examples/0001_q000_m01_k00_force_ub3lyp_def2tzvpd.fchk")

res_basis = from_iodata(res_mol)
unr_basis = from_iodata(unr_mol)

In [50]:
res_mo = res_mol.mo
unr_mo = unr_mol.mo

In [51]:
res_mol

IOData(atcharges={'mulliken': array([-0.84854995,  0.2654845 ,  0.22719457,  0.23939484,  0.5827754 ,
       -0.58437473, -0.13729228,  0.19723758, -0.72592967,  0.25887026,
        0.24362204,  0.28156745]), 'esp': array([-0.87277124,  0.19923423,  0.24368374,  0.24687887,  0.73152104,
       -0.56145798, -0.33180084,  0.27919542, -0.39843488,  0.15888804,
        0.15312703,  0.15193658]), 'npa': array([-0.68372356,  0.19705745,  0.23398906,  0.23391084,  0.63539692,
       -0.6186563 , -0.58794912,  0.38808407, -0.40435125,  0.21138167,
        0.19928245,  0.19557775]), 'mbs': array([-0.28636364,  0.08295574,  0.1214883 ,  0.12172806,  0.441787  ,
       -0.53158836, -0.42040325,  0.27433022, -0.13535692,  0.12201555,
        0.11083254,  0.09857476]), 'hirshfeld': array([-0.08932595,  0.03141634,  0.04748376,  0.04719025,  0.16083449,
       -0.31900037, -0.08285949,  0.12004421, -0.03202121,  0.04017923,
        0.03748519,  0.03856857]), 'cm5': array([-0.22787345,  0.08758915,  

In [52]:
res_rdm_a = (res_mol.mo.coeffsa * res_mol.mo.occsa) @ res_mol.mo.coeffsa.conj().T
res_rdm_b = (res_mol.mo.coeffsb * res_mol.mo.occsb) @ res_mol.mo.coeffsb.conj().T

unr_rdm_a = (unr_mol.mo.coeffsa * unr_mol.mo.occsa) @ unr_mol.mo.coeffsa.conj().T
unr_rdm_b = (unr_mol.mo.coeffsb * unr_mol.mo.occsb) @ unr_mol.mo.coeffsb.conj().T

In [32]:
oned = GaussChebyshev(npoints=100)
rgrid = BeckeRTransform(0.0, R=1.5).transform_1d_grid(oned)

res_grid = MolGrid.from_preset(
    atnums=res_mol.atnums,
    atcoords=res_mol.atcoords,
    rgrid=rgrid,
    preset="fine",
    aim_weights=BeckeWeights(),
    store=True,
)

unr_grid = MolGrid.from_preset(
    atnums=unr_mol.atnums,
    atcoords=unr_mol.atcoords,
    rgrid=rgrid,
    preset="fine",
    aim_weights=BeckeWeights(),
    store=True,
)

res_dens_a = evaluate_density(res_rdm_a, res_basis, res_grid.points)
res_dens_b = evaluate_density(res_rdm_b, res_basis, res_grid.points)
res_dens_total = res_dens_a + res_dens_b

unr_dens_a = evaluate_density(unr_rdm_a, unr_basis, unr_grid.points)
unr_dens_b = evaluate_density(unr_rdm_b, unr_basis, unr_grid.points)

C:\Users\ccast\modules\grid\src\grid\atomgrid.py:907: UserWarning: Lebedev weights are negative which can introduce round-off errors.
  sphere_grid = AngularGrid(degree=deg_i, method=method)
C:\Users\ccast\modules\grid\src\grid\atomgrid.py:907: UserWarning: Lebedev weights are negative which can introduce round-off errors.
  sphere_grid = AngularGrid(degree=deg_i, method=method)


In [55]:
res_grid.integrate(res_dens_total), unr_grid.integrate(unr_dens_a + unr_dens_b)

(np.float64(39.999857206352665), np.float64(39.99985720590134))

In [54]:
print(np.allclose(res_dens_a, unr_dens_a))
print(np.allclose(res_dens_b, unr_dens_b))

True
True


### MBIS PARTITIONING

In [61]:
from denspart.cache import ComputeCache
from denspart.mbis import MBISProModel
from denspart.vh import optimize_reduce_pro_model
from denspart.properties import safe_ratio
mbis_model_init = MBISProModel.from_geometry(res_mol.atnums, res_mol.atcoords)


In [62]:
mbis_model, localgrids = optimize_reduce_pro_model(
                mbis_model_init,
                res_grid,
                res_dens_total,
                1e-8,
                1000,
                1e-20,
                ComputeCache(),
            )

Building local grids
Integral of density: 39.999857206352665
Optimization
#Iter  #Call         ekld          kld  -constraint     grad.rms  cputime (s)
-----  -----  -----------  -----------  -----------  -----------  -----------
    1      1    1.2578350    1.2576922   1.4279e-04   1.9257e-01    0.0468750
    2      2    0.7743920    0.3102369   4.6416e-01   1.0137e-01    0.0156250
    3      3    0.8892965   -0.9287651   1.8181e+00   2.3762e-01    0.0156250


c:\Users\ccast\anaconda3\envs\dev\Lib\site-packages\scipy\optimize\_trustregion_constr\minimize_trustregion_constr.py:341: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  modified_lb = np.nextafter(bounds.lb, -np.inf, where=bounds.lb > -np.inf)
c:\Users\ccast\anaconda3\envs\dev\Lib\site-packages\scipy\optimize\_trustregion_constr\minimize_trustregion_constr.py:342: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  modified_ub = np.nextafter(bounds.ub, np.inf, where=bounds.ub < np.inf)


    4      4    0.5646797   -1.1601662   1.7248e+00   7.9524e-02    0.0156250
    5      5    0.5409756   -1.1136666   1.6546e+00   6.8229e-02    0.0156250
    6      5    0.5409756   -1.1136666   1.6546e+00   6.8229e-02    0.0156250
    7      6    0.4352845   -0.3193861   7.5467e-01   3.7742e-02    0.0156250
    8      7    0.3958538    0.3741692   2.1685e-02   1.5203e-02    0.0312500
    9      8    0.3944240    0.3192451   7.5179e-02   1.5593e-02    0.0156250
   10      9    0.3857180    0.2754924   1.1023e-01   1.6041e-02    0.0312500
   11     10    0.3774544    0.1649980   2.1246e-01   1.6775e-02    0.0312500
   12     10    0.3774544    0.1649980   2.1246e-01   1.6775e-02    0.0312500
   13     11    0.3685964    0.3642886   4.3078e-03   5.5740e-03    0.0312500
   14     12    0.3663435    0.2429429   1.2340e-01   5.1072e-03    0.0156250
   15     13    6.8830140    8.3660182  -1.4830e+00   1.5849e+00    0.0312500
   16     14    3.4681900    3.5178323  -4.9642e-02   2.9450e-01

In [90]:
pro = mbis_model.compute_density(res_grid, localgrids)
stockholder_weights = []

for iatom in range(mbis_model.natom):
    pro_atom = mbis_model.compute_proatom(
        iatom, res_grid.points
    )

    weight = safe_ratio(pro_atom, pro)
    stockholder_weights.append(weight)

stockholder_weights = np.array(stockholder_weights)

In [91]:
len(stockholder_weights), len(res_mol.atnums)

(12, 12)

In [105]:
# Stockholder weights for atom 0
w0 = stockholder_weights[0]

# Verify normalization
print(stockholder_weights.sum(axis=0))

[1. 1. 1. ... 0. 0. 0.]


In [104]:
# at each point, the sum of the weights should be 1.0 test it:
for j in range(len(stockholder_weights[0])):
    sum_weights = 0
    for i in range(len(stockholder_weights)):
        sum_weights+=stockholder_weights[i][j]
    if np.isclose(sum_weights, 1.0):
        print(sum_weights)
    elif not np.isclose(sum_weights, 1.0):
        print("Sum of weights is not 1.0 at point", j, ":", sum_weights)

0.9999999999999999
1.0000000000000002
1.0000000000000002
0.9999999999999997
1.0000000000000002
1.0
1.0000000000000002
0.9999999999999999
1.0
1.0
1.0
1.0
1.0000000000000002
0.9999999999999997
0.9999999999999999
1.0000000000000002
1.0000000000000002
0.9999999999999999
1.0
0.9999999999999996
1.0000000000000004
1.0000000000000002
0.9999999999999998
1.0000000000000002
1.0000000000000002
1.0
1.0
1.0000000000000002
1.0000000000000002
1.0000000000000007
1.0
0.9999999999999999
0.9999999999999997
1.0000000000000002
0.9999999999999998
0.9999999999999999
1.0
1.0
0.9999999999999998
1.0000000000000002
1.0
1.0
1.0
0.9999999999999999
0.9999999999999999
0.9999999999999998
0.9999999999999999
0.9999999999999998
1.0000000000000004
1.0000000000000002
1.0
0.9999999999999999
0.9999999999999998
0.9999999999999999
1.0000000000000004
1.0000000000000004
1.0
1.0
0.9999999999999996
1.0
0.9999999999999998
1.0000000000000002
1.0000000000000004
1.0
1.0000000000000002
1.0000000000000002
0.9999999999999998
1.0000000000

### Analysis and test of Stockholder ratio

In [ ]:
# TASK: Compute the charges with the stockholder weights and the molecular density. 
# The charges should sum to the total number of electrons in the molecule. 
# The charges can be computed by integrating the product of the stockholder weights and the molecular density over the grid points for each atom.
# Compare the computed charges with the charges from the mbis_model. The charges from the mbis_model can be accessed using:
mbis_model.charges

array([-0.65388816,  0.1546091 ,  0.19088538,  0.191371  ,  0.78550883,
       -0.61758965, -0.57910633,  0.3461017 , -0.15600121,  0.12326306,
        0.11093867,  0.10405026])

### Taylor Expansion Approximation of Exchange/Fermi Hole:

Apply the variable transformation $\textbf{r} = \textbf{r}_1$ and $\textbf{s} = \textbf{r}_2 - \textbf{r}_1$. The approxmiation of the Fermi hole is a spherical average over $\textbf{s}$ about the reference point $\textbf{r}$

$$
h_{X\sigma}(\textbf{r}, \textbf{r+s}) = - \frac{|\rho_\sigma(\textbf{r}, \textbf{r+s})|^2}{\rho_\sigma(\textbf{r})} \approx - \rho_\sigma(\textbf{r}) - Q_\sigma(\textbf{r}) s^2.
$$

where 

$$
Q_\sigma(\textbf{r}) = \frac{1}{6} \left[ \nabla^2 \rho_\sigma(\textbf{r}) - 2\tau_\sigma + \frac{1}{2} \frac{(\nabla\rho_\sigma(\textbf{r}))^2}{\rho_\sigma(\textbf{r})} \right]
$$

Because the exchange energy only depends on the spherical average of the Fermi hole, the approximation of the Fermi hole is restricted to spherically symmetric models.

The probability of finding another same-spin electron at the same position as the reference electron is completely extingushed
$$
h_{X\sigma}(\textbf{r}, \textbf{r}) = - \rho_\sigma(\textbf{r})
$$

the exchange hole at any reference point in any inhomogenenous system can be calculated by taking the spherically averaged exchange hole of the 1s electron through a parameterization.

...
...
...

$a$ and $b$ can be solved for by finding $x=ab$ that satisfies the nonlinear one-dimensional equation 

$$
\frac{xe^{-2x/3}}{x-2} = \frac{2}{3} \pi^{2/3} \frac{\rho_\sigma^{5/3}}{Q_\sigma}
$$

$b$ is then given by

$$
b^3 = \frac{x^3e^{-x}}{8 \pi \rho_\sigma}
$$

and $a = x/b$

In [ ]:
mol = load_one("../examples/water.fchk")
rdm = mol.one_rdms.get("scf")
basis = from_iodata(mol)
mo = mol.mo
rdm_alpha = (mo.coeffsa * mo.occsa) @ mo.coeffsa.conj().T
rdm_beta = (mo.coeffsb * mo.occsb) @ mo.coeffsb.conj().T

oned = GaussChebyshev(npoints=30)
rgrid = BeckeRTransform(0.0, R=1.5).transform_1d_grid(oned)

grid = MolGrid.from_size(
    atnums=mol.atnums,
    atcoords=mol.atcoords,
    rgrid=rgrid,
    size=38,
    aim_weights=BeckeWeights(),
    store=True,
)

dens_alpha = evaluate_density(rdm_alpha, basis, grid.points)
grad_alpha = evaluate_density_gradient(rdm_alpha, basis, grid.points)
lap_alpha = evaluate_density_laplacian(rdm_alpha, basis, grid.points)
tau_alpha = evaluate_posdef_kinetic_energy_density(rdm_alpha, basis, grid.points)

dens_beta = evaluate_density(rdm_beta, basis, grid.points)
grad_beta = evaluate_density_gradient(rdm_beta, basis, grid.points)
lap_beta = evaluate_density_laplacian(rdm_beta, basis, grid.points)
tau_beta = evaluate_posdef_kinetic_energy_density(rdm_beta, basis, grid.points)

/home/merreeves/ffprime/.ffprime/lib/python3.12/site-packages/grid/molgrid.py:436: RuntimeWarning: Sizes are used for making the angular grids, degrees are ignored!
  AtomGrid(rad_grid, degrees=None, sizes=[size], center=atcoord, rotate=rotate)


In [4]:
from ffprime.nonbonding.vdw import XDMParameters

xdm_test = XDMParameters(None, None, None)
a_alpha, b_alpha = xdm_test.compute_hole_parameters(dens_alpha, grad_alpha, lap_alpha, tau_alpha)
a_beta, b_beta = xdm_test.compute_hole_parameters(dens_beta, grad_beta, lap_beta, tau_beta)

print("Alpha hole parameters shape: a =", a_alpha.shape, ", b =", b_alpha.shape)
print("Beta hole parameters shape: a =", a_beta.shape, ", b =", b_beta.shape)

print("Maximum a_alpha:", np.max(a_alpha))
print("Maximum b_alpha:", np.max(b_alpha))
print("Maximum a_beta:", np.max(a_beta))
print("Maximum b_beta:", np.max(b_beta))

print("Minimum a_alpha:", np.min(a_alpha))
print("Minimum b_alpha:", np.min(b_alpha))
print("Minimum a_beta:", np.min(a_beta))
print("Minimum b_beta:", np.min(b_beta))

Alpha hole parameters shape: a = (3420,) , b = (3420,)
Beta hole parameters shape: a = (3420,) , b = (3420,)
Maximum a_alpha: 30.33528726164372
Maximum b_alpha: 9.548942728281117
Maximum a_beta: 30.33528726164372
Maximum b_beta: 9.548942728281117
Minimum a_alpha: 1.3701914626601406
Minimum b_alpha: 0.06592981905033164
Minimum a_beta: 1.3701914626601406
Minimum b_beta: 0.06592981905033164
